# 第09讲 gpt2_train_tokenizer.ipynb

按原单元编号阅读。配套讲解见本讲的解释.md，运行方法与记录见复现目录。


## 单元 0（markdown）：训练tokenizer入口

从代码文本学习新的BPE词表与merges，不训练Transformer。


## tokenizer

## 单元 1（code）：导入加载、语料与进度接口

AutoTokenizer是transformers的类；load_dataset是datasets函数；tqdm是可迭代对象的进度包装器，不负责数据本身或训练算法。导入不读语料。


In [1]:
from transformers import AutoTokenizer
from datasets import load_dataset
from tqdm import tqdm

## 单元 2（code）：原本地codeparrot路径

load_dataset(./codeparrot/,split=train,streaming=True)相对当前工作目录；返回流式IterableDataset时按需读。原仓库没有本地语料，不能仅凭streaming生成文件。原格保留但未运行；独立补充用临时JSONL与真实datasets json读取，不假冒课程语料。


In [2]:
dataset = load_dataset('./codeparrot/', split='train', streaming=True)

Resolving data files:   0%|          | 0/184 [00:00<?, ?it/s]

## 单元 3（code）：建立有消费位置的迭代器

iter(dataset)返回迭代器，next每调用一次消费一条dict；iter_dataset跨后续函数共用，没有自动复位。本次在明确小语料IterableDataset上下文执行原句。


In [3]:
iter_dataset = iter(dataset)

## 单元 4（code）：旧GPT2 tokenizer作为处理模板

加载原GPT2 fast tokenizer；后面重建词表不会保留原ID语义。本地路径补充仅改字符串/local_files_only，固定revision且无权重。


In [4]:
tokenizer = AutoTokenizer.from_pretrained('gpt2')

## 单元 5（markdown）：构建alphabet与训练批标题

先准备256个基础符号，再提供字符串批给trainer。


### build

## 单元 6（code）：字节正反映射与基础词表

无参bytes_to_unicode返回256项dict；(v,k)反转items，list(keys)收集映射字符，base_vocab为list[str]长度256，不是完整50257项词表。initial_alphabet用于覆盖任意UTF8字节。


In [5]:
from transformers.models.gpt2.tokenization_gpt2 import bytes_to_unicode
bytes_to_unicode_map = bytes_to_unicode()
unicode_to_bytes_map = dict((v, k) for k, v in bytes_to_unicode_map.items())
base_vocab = list(unicode_to_bytes_map.keys())

## 单元 7（code）：length与batch_iterator

length=100000设全局读取上限；def函数默认batch_size=1000，读取全局iter_dataset/length。yield使调用返回惰性生成器，每批为list[str]；两个_是普通循环变量。range(0,length,batch_size)按批数计tqdm，内部next在不满批时会耗尽并触发RuntimeError。不能整除时仍读满末批，可能超读；本次执行原函数体并用小上下文实测，不消费10万条。


In [6]:
length = 100000
def batch_iterator(batch_size=1000):
#     for _ in tqdm(range(0, len(dataset), batch_size)):
    for _ in tqdm(range(0, length, batch_size)):
        yield [next(iter_dataset)['content'] for _ in range(batch_size)]

## 单元 8（code）：训练新BPE tokenizer

fast对象train_new_from_iterator接收batch_iterator()、vocab_size=12500和initial_alphabet=base_vocab。返回新对象new_tokenizer，不修改旧tokenizer；清空旧vocab/merges并保留处理/特殊token配置。目标含alphabet/特殊token，语料不足可能达不到12500；原格未原配置运行，补充仅改vocab320并调用真实原生成器（默认批大小仍1000）。


In [7]:
# 比较耗时
new_tokenizer = tokenizer.train_new_from_iterator(batch_iterator(), 
                                                  vocab_size=12500, 
                                                  initial_alphabet=base_vocab)

100%|██████████| 100/100 [00:24<00:00,  4.10it/s]


## 单元 9（code）：展示新对象

读取new_tokenizer repr，显示的是训练后的词表/配置，不涉及模型logits。


In [8]:
new_tokenizer

GPT2TokenizerFast(name_or_path='gpt2', vocab_size=12500, model_max_length=1024, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, clean_up_tokenization_spaces=True)

## 单元 10（markdown）：新词表检查标题

检查实际大小、特殊token和合并规则，不仅看目标vocab_size。


### new tokenizer

## 单元 11（code）：按整数ID升序排序

vocab.items是(token,ID)，sorted key=lambda x:x[1]升序、reverse=False；返回新list tokens，不改变tokenizer词表。


In [11]:
tokens = sorted(new_tokenizer.vocab.items(), key=lambda x: x[1], reverse=False)

## 单元 12（code）：观察alphabet前10项

切片[:10]不含右边界，读取映射字典插入顺序的前10显示符号，非最常见的10个token。


In [17]:
base_vocab[:10]

['!', '"', '#', '$', '%', '&', "'", '(', ')', '*']

## 单元 13（code）：第257–279项显示转换

tokens[257:280]最多23个二元组，for t,_解包忽略ID，转换单token为显示文本并返回list[(原token,显示文本)]。257只是原代码切分假设，应查实际基础/特殊token排布；单token可能不是完整UTF8字符。


In [30]:
# 257:280
[(t, new_tokenizer.convert_tokens_to_string([t])) for t, _ in tokens[257:280]]

[('ĠĠ', '  '),
 ('ĠĠĠĠ', '    '),
 ('ĠĠĠ', '   '),
 ('ĠĠĠĠĠĠĠĠ', '        '),
 ('se', 'se'),
 ('in', 'in'),
 ('ĠĠĠĠĠĠĠ', '       '),
 ('re', 're'),
 ('on', 'on'),
 ('te', 'te'),
 ('ĊĠĠĠĠĠĠĠ', '\n       '),
 ('ĊĠĠĠĠĠĠĠĠ', '\n        '),
 ('or', 'or'),
 ('st', 'st'),
 ('de', 'de'),
 ('ĊĠĠĠ', '\n   '),
 ('th', 'th'),
 ('le', 'le'),
 ('Ġ=', ' ='),
 ('lf', 'lf'),
 ('self', 'self'),
 ('me', 'me'),
 ('al', 'al')]

## 单元 14（code）：词表尾15项

[-15:]从倒数第15开始，按前面ID升序列表取尾部；不是按频率排序。转换只是可视化，不撤销词表训练。


In [33]:
# last 15
[(t, new_tokenizer.convert_tokens_to_string([t])) for t, _ in tokens[-15:]]

[('ĠSetup', ' Setup'),
 ('publisher', 'publisher'),
 ('DER', 'DER'),
 ('Ġcapt', ' capt'),
 ('Ġembedded', ' embedded'),
 ('Ġregarding', ' regarding'),
 ('Bundle', 'Bundle'),
 ('355', '355'),
 ('Ġrecv', ' recv'),
 ('Ġdmp', ' dmp'),
 ('Ġvault', ' vault'),
 ('ĠMongo', ' Mongo'),
 ('Ġpossibly', ' possibly'),
 ('implementation', 'implementation'),
 ('Matches', 'Matches')]

## 单元 15（markdown）：Python关键字检查标题

检查裸字符串是否词表项，只是覆盖统计，不是代码生成质量。


### python keyword

## 单元 16（code）：导入标准库keyword

keyword.kwlist是当前Python版本硬关键字列表。软关键字另有softkwlist，不能硬编码某版数量或混淆两类。


In [34]:
# python 标准库
import keyword

## 单元 17（code）：关键字数量

len(keyword.kwlist)返回int；本次Python3.12有35项，此数量依版本可能变，不是目标词表大小。


In [35]:
len(keyword.kwlist)

35

## 单元 18（code）：检查裸关键字成员关系

for遍历每个kw字符串，kw not in new_tokenizer.vocab检查字典键，不存在时print。带空格的Ġreturn和裸return是不同token；缺失不意味着模型不能用多项表示，也不证明生成能力。


In [39]:
for kw in keyword.kwlist:
    if kw not in new_tokenizer.vocab:
        print(f'`{kw}` not in the new tokenizer')

`await` not in the new tokenizer
`finally` not in the new tokenizer
`nonlocal` not in the new tokenizer


## 单元 19（markdown）：第二轮训练标题

retrain这里是再次从旧tokenizer模板训练新的词表，不是继续更新第8格的新merges。


### retrain

## 单元 20（code）：第二轮重定义函数并继续消费

length=100000*2是200000；同名def覆盖旧函数，依旧共享iter_dataset，没有iter(dataset)复位。train_new_from_iterator在旧tokenizer上调用且vocab32768=2^15，同时改变样本范围/规模与预算，不能归因单一变量。本次AST审计的小语料补充仅改length和vocab，仍真实执行原生成器默认1000，证明继续消费。


In [40]:
length = 100000*2
def batch_iterator(batch_size=1000):
    for _ in tqdm(range(0, length, batch_size)):
        yield [next(iter_dataset)['content'] for _ in range(batch_size)]
new_tokenizer = tokenizer.train_new_from_iterator(batch_iterator(), 
                                                  vocab_size=32768, 
                                                  initial_alphabet=base_vocab)

100%|██████████| 200/200 [00:48<00:00,  4.15it/s]


## 单元 21（code）：再次检查关键字

对第二轮新对象重复裸词表成员检查；与第一轮相同API，打印差异不等于下游指标提升。


In [41]:
for kw in keyword.kwlist:
    if kw not in new_tokenizer.vocab:
        print(f'`{kw}` not in the new tokenizer')

`nonlocal` not in the new tokenizer


## 单元 22（markdown）：Hub发布标题

保存本地与推送远程是两种操作；本讲验证采用临时目录save_pretrained/重载。


### to the hub

## 单元 23（code）：创建仓库提示为注释

两条#行不会导入huggingface_hub或create_repo。实际创建需账号/权限，原notebook并未在此格执行。用户授权的是公开学习GitHub仓库，不是讲师Hugging Face组织。


In [45]:
# from huggingface_hub import create_repo
# create_repo('codeparrot')

RepoUrl('https://huggingface.co/lanchunhui/codeparrot', endpoint='https://huggingface.co', repo_type='model', repo_id='lanchunhui/codeparrot')

## 单元 24（code）：原代理环境写入

os.environ映射赋值HTTP/HTTPS代理到127.0.0.1:7890，会改变本进程后续请求。必须有该服务才有效；本次明确跳过，不修改真实代理。


In [51]:
import os
os.environ['HTTP_PROXY'] = 'http://127.0.0.1:7890'
os.environ['HTTPS_PROXY'] = 'http://127.0.0.1:7890'

## 单元 25（code）：原讲师组织Hub上传

ckpt/org为字符串，new_tokenizer.push_to_hub会向远程写文件，organization为历史API用法。上传会将词表文件保存到Hub仓库；若先在本地查看，可用save_pretrained保存，再用AutoTokenizer重载。


In [52]:
ckpt = 'codeparrot'
org = 'lanchunhui'
new_tokenizer.push_to_hub(ckpt, organization=org)

CommitInfo(commit_url='https://huggingface.co/lanchunhui/codeparrot/commit/212e703fb744884f9563cfbf6de94ffd5792606a', commit_message='Upload tokenizer', commit_description='', oid='212e703fb744884f9563cfbf6de94ffd5792606a', pr_url=None, pr_revision=None, pr_num=None)

## 单元 26（code）：空代码格

原空格保留，注释版AST空。
